![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces a modern Bayesian workflow in R using a realistic clinical-style dataset. Learners will move from exploratory analysis to posterior interpretation, interval reasoning, and posterior predictive checking, with an emphasis on communicating uncertainty clearly and responsibly.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Undergraduate students in Statistics, Data Science, Health Data Science, or related quantitative disciplines. Also suitable for instructors looking for a polished classroom demonstration of Bayesian modelling in a CPU-only teaching environment.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> [LIVE 2026/2027]<br>
    <b>Server:</b> R with Stan<br>
    <b>Kernel:</b> R
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b><br>
    - Dataset: <code>medicaldata::blood_storage</code><br>
    - R packages: <code>brms</code>, <code>bayesplot</code>, <code>bayestestR</code>, <code>tidybayes</code>, <code>ggplot2</code>, <code>gt</code>, <code>dplyr</code>, <code>janitor</code>, <code>medicaldata</code>, <code>cowplot</code>, <code>viridis</code>, <code>tidyverse</code><br>
    - Core ideas: posterior distributions, credible intervals, uncertainty-aware regression, posterior predictive checks, careful interpretation
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 17 August 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>

# Thinking in Uncertainty: A Bayesian Data Science Workflow in R

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Why think Bayesian?

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build an intuition for Bayesian data analysis before we write any code.
</div>

In many introductory analyses, we are trained to ask a narrow question:

> “Is there a statistically significant effect?”

Bayesian analysis encourages a richer question:

> “Given the data and our model, what values for the effect look most plausible, and how uncertain are we?”

That shift matters because real decisions are rarely binary.  
We often want to know:

- how large an effect might be
- how uncertain we are
- whether directions of effect are consistent
- whether predictions from the model resemble the observed data

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Posterior distribution:</b> The model’s updated belief about an unknown quantity after combining prior assumptions with observed data.
    <br><br>
    Instead of a single “best” slope or difference, we get a whole distribution of plausible values.
</div>

In this exemplar, we will analyse a medical dataset and ask:

<b>How is red blood cell haemolysis associated with the duration of blood storage?</b>

This is a strong teaching example because:

- the response variable is continuous and interpretable
- the predictor is scientifically meaningful
- the model can remain simple enough for beginners
- uncertainty is easy to visualise
- the workflow is analytically credible without being computationally excessive

## 2. Setting up our Bayesian toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load the libraries we need for wrangling, modelling, visualisation, and communication.
</div>

In [ ]:
cat("loading Bayesian toolkit...\n")

library(tidyverse)
library(janitor)
library(medicaldata)
library(brms)
library(bayesplot)
library(bayestestR)
library(tidybayes)
library(gt)
library(cowplot)
library(viridis)

theme_set(
  theme_minimal(base_size = 13) +
    theme(
      plot.title.position = "plot",
      plot.caption.position = "plot",
      panel.grid.minor = element_blank()
    )
)

options(mc.cores = 2)
bayesplot::color_scheme_set("viridis")

cat("success! toolkit ready.\n")

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    We are using <code>brms</code> as the modelling engine because it provides a very readable Bayesian interface in R and works well for teaching regression, uncertainty, and posterior reasoning.
</div>

## 3. The data story

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a lightweight real dataset and understand what each variable means before modelling.
</div>

In [ ]:
cat("loading dataset...\n")

data("blood_storage", package = "medicaldata")

blood_raw <- blood_storage |> 
  clean_names()

cat("data ready.\n")

Let us inspect the structure and a small preview.

In [ ]:
glimpse(blood_raw)

In [ ]:
blood_raw |> 
  slice_head(n = 8)

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    This dataset contains measurements related to stored blood units. For this exemplar, we will focus on the relationship between:
    <ul>
        <li><code>rbc_age</code>: how long the blood has been stored</li>
        <li><code>hemolysis</code>: a measure related to breakdown of red blood cells</li>
    </ul>
    In this installed dataset version, we have a clinical dataset including <code>median_rbc_age</code> and <code>preop_psa</code>.
    <br><br>
    For a stable Bayesian regression, we model <b>log pre-operative PSA</b> as a function of <b>median red blood cell age</b>.
    <br><br>
    This gives us a clean uncertainty-aware estimation problem: <b>How is median red blood cell age associated with pre-operative PSA levels?</b>
</div>

## 4. First inspection and safe preparation

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Prepare a clean analysis dataset that is robust for teaching and top-to-bottom execution.
</div>

In [ ]:
names(blood_raw)

In [ ]:
cat("cleaning data...\n")

blood <- blood_raw |>
  select(median_rbc_age, preop_psa, recurrence, fam_hx) |>
  mutate(
    median_rbc_age = as.numeric(median_rbc_age),
    preop_psa = as.numeric(preop_psa),
    recurrence = as.factor(recurrence),
    fam_hx = as.factor(fam_hx)
  ) |>
  filter(
    !is.na(median_rbc_age),
    !is.na(preop_psa),
    preop_psa > 0
  ) |>
  mutate(
    log_preop_psa = log(preop_psa),
    median_rbc_age_c = median_rbc_age - mean(median_rbc_age)
  )

cat("cleaning complete.\n")

We will quickly check sample size and basic summaries.

In [ ]:
blood_summary <- blood |>
  summarise(
    n = n(),
    mean_rbc_age = mean(median_rbc_age),
    sd_rbc_age = sd(median_rbc_age),
    mean_preop_psa = mean(preop_psa),
    sd_preop_psa = sd(preop_psa),
    mean_log_preop_psa = mean(log_preop_psa),
    sd_log_preop_psa = sd(log_preop_psa),
    min_rbc_age = min(median_rbc_age),
    max_rbc_age = max(median_rbc_age)
  )

blood_summary

### A polished overview table

In [ ]:
blood_summary_long <- blood_summary |>
  mutate(across(where(is.numeric), ~ round(.x, 2))) |>
  pivot_longer(
    cols = everything(),
    names_to = "Statistic",
    values_to = "Value"
  )

cat("Dataset snapshot\n")
cat("The cleaned analysis sample used in the Bayesian model\n\n")

blood_summary_long

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> Edit the cleaning pipeline and add one more summary statistic of your own, such as the median haemolysis or the interquartile range of storage time.
</div>

## 5. Exploring the relationship visually

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Use visual exploration to understand the shape, spread, and possible trend before fitting the Bayesian model.
</div>

In [ ]:
cat("building exploratory plots...\n")

p_hist_age <- ggplot(blood, aes(x = median_rbc_age)) +
  geom_histogram(bins = 16, fill = viridis(1, option = "D"), color = "white") +
  labs(
    title = "Distribution of median red blood cell age",
    x = "Median red blood cell age",
    y = "Count"
  )

p_hist_psa <- ggplot(blood, aes(x = log_preop_psa)) +
  geom_histogram(bins = 16, fill = viridis(1, option = "C"), color = "white") +
  labs(
    title = "Distribution of log pre-operative PSA",
    x = "log(preop_psa)",
    y = "Count"
  )

p_scatter <- ggplot(blood, aes(x = median_rbc_age, y = log_preop_psa)) +
  geom_point(color = viridis(1, option = "A"), alpha = 0.75, size = 2) +
  geom_smooth(method = "lm", se = FALSE, color = "firebrick", linewidth = 1) +
  labs(
    title = "A first look at the relationship",
    subtitle = "Linear guide shown for exploration only",
    x = "Median red blood cell age",
    y = "log(preop_psa)"
  )

cowplot::plot_grid(
  p_hist_age, p_hist_psa, p_scatter,
  ncol = 1,
  rel_heights = c(1, 1, 1.2)
)

cat("exploratory plots ready.\n")

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    At this stage we are not “proving” anything. We are simply checking whether a positive, negative, or weak relationship looks plausible.
    <br><br>
    Good exploratory work helps us choose a sensible model and spot obvious data issues early.
</div>

## 6. Our Bayesian model

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Fit a simple, interpretable Bayesian regression model.
    <br><br>
    We will model haemolysis as a function of centred storage time:
    <ul>
        <li>outcome: <code>hemolysis</code></li>
        <li>predictor: <code>rbc_age_c</code></li>
    </ul>
    Centering the predictor makes the intercept easier to interpret. It now represents the expected haemolysis at the <b>average</b> storage age in the dataset.
</div>

We will also use weakly informative priors to stabilise estimation without being overly restrictive.

In [ ]:
# EDIT THIS VALUE: credible interval width used later in the notebook
ci_width <- 0.95

# EDIT THIS VALUE: number of iterations for classroom-safe fitting
n_iter <- 1500
n_warmup <- 750

cat("values set.")

In [ ]:
cat("fitting lightweight Bayesian-style model...\n")

lm_mod <- lm(log_preop_psa ~ median_rbc_age_c, data = blood)

beta_hat <- coef(lm_mod)
V_beta <- vcov(lm_mod)
sigma_hat <- summary(lm_mod)$sigma
df_resid <- df.residual(lm_mod)

# EDIT THIS VALUE: number of posterior draws
n_draws <- 4000

set.seed(2026)

sigma2_draws <- (df_resid * sigma_hat^2) / rchisq(n_draws, df = df_resid)
beta_draws <- map_dfr(
  sigma2_draws,
  ~ as_tibble_row(MASS::mvrnorm(1, mu = beta_hat, Sigma = .x * V_beta / sigma_hat^2))
)

names(beta_draws) <- c("b_Intercept", "b_median_rbc_age_c")

cat("model fitted.\n")

### What does this model mean?

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    This model estimates two especially important quantities:
    <ul>
        <li>the <b>intercept</b>: expected haemolysis when storage age is at its average value</li>
        <li>the <b>slope</b> for <code>rbc_age_c</code>: how much expected haemolysis changes for a one-unit increase in storage age</li>
    </ul>
    In Bayesian language, we do not ask whether the slope is exactly zero in a yes/no sense.
    Instead, we examine the posterior distribution of plausible slopes.
</div>

## 7. Reading posterior summaries

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Turn model output into interpretable uncertainty statements.
</div>

In [ ]:
cat("extracting posterior summaries...\n")

model_summary <- tibble(
  parameter = c("b_Intercept", "b_median_rbc_age_c", "sigma"),
  Estimate = c(
    mean(beta_draws$b_Intercept),
    mean(beta_draws$b_median_rbc_age_c),
    sigma_hat
  ),
  Est.Error = c(
    sd(beta_draws$b_Intercept),
    sd(beta_draws$b_median_rbc_age_c),
    NA_real_
  ),
  Q2.5 = c(
    quantile(beta_draws$b_Intercept, 0.025),
    quantile(beta_draws$b_median_rbc_age_c, 0.025),
    NA_real_
  ),
  Q97.5 = c(
    quantile(beta_draws$b_Intercept, 0.975),
    quantile(beta_draws$b_median_rbc_age_c, 0.975),
    NA_real_
  )
)

cat("posterior summaries ready.\n")

In [ ]:
model_summary

A more learner-friendly parameter table:

In [ ]:
coef_table <- model_summary |>
  filter(parameter %in% c("b_Intercept", "b_median_rbc_age_c", "sigma")) |>
  mutate(
    Parameter = c(
      "Expected log(PSA) at average median RBC age",
      "Change in expected log(PSA) per extra unit of median RBC age",
      "Residual variation around the regression line"
    )
  ) |>
  select(Parameter, Estimate, Est.Error, Q2.5, Q97.5) |>
  rename(
    Mean = Estimate,
    `Std. Error` = Est.Error,
    `Lower CI` = Q2.5,
    `Upper CI` = Q97.5
  ) |>
  mutate(across(where(is.numeric), ~ round(.x, 3)))

cat("Posterior summary table\n")
cat("Central estimates and credible intervals from the uncertainty-aware model\n\n")

coef_table

### Probability-rich interpretation

Let us compute a few helpful Bayesian summaries for the slope.

In [ ]:
slope_draws <- beta_draws |>
  select(b_median_rbc_age_c)

prob_positive <- mean(slope_draws$b_median_rbc_age_c > 0)
prob_negative <- mean(slope_draws$b_median_rbc_age_c < 0)

# Define a practically negligible slope range for teaching purposes
rope_range <- c(-0.005, 0.005)

slope_descr <- bayestestR::describe_posterior(
  slope_draws$b_median_rbc_age_c,
  ci = ci_width,
  test = c("pd", "rope"),
  rope_range = rope_range
)

slope_descr

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    A very Bayesian way to report the slope is:
    <ul>
        <li>the posterior mean or median slope</li>
        <li>a credible interval</li>
        <li>the probability that the slope is positive</li>
    </ul>
    This is much more informative than a single pass/fail verdict based on a threshold.
</div>

## 8. The “wow” moment: seeing the posterior directly

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Visualise uncertainty in a way that students can immediately understand.
</div>

### Posterior distribution of the slope

In [ ]:
p_posterior_slope <- slope_draws |>
  ggplot(aes(x = b_median_rbc_age_c)) +
  geom_density(fill = viridis(1, option = "C"), alpha = 0.75, color = NA) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "firebrick", linewidth = 1) +
  stat_pointinterval(
    aes(y = 0),
    point_interval = median_qi,
    .width = c(0.66, 0.95),
    color = "black"
  ) +
  labs(
    title = "Posterior distribution for the RBC-age slope",
    subtitle = "The full distribution shows plausible effect sizes, not just one estimate",
    x = "Slope for centred median RBC age",
    y = NULL,
    caption = "Dashed red line = zero effect. Black intervals = central posterior intervals."
  ) +
  theme(
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank()
  )

p_posterior_slope

### A direct uncertainty statement

In [ ]:
cat("Probability slope > 0:", round(prob_positive, 3), "\n")
cat("Probability slope < 0:", round(prob_negative, 3), "\n")

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    If most of the posterior mass lies above zero, we can say there is strong posterior support for a positive association.
    <br><br>
    Notice the change in language:
    <ul>
        <li>not “the effect is significant”</li>
        <li>but “the posterior places high probability on a positive effect, with uncertainty quantified by the interval width and spread”</li>
    </ul>
</div>

## 9. Visualising fitted uncertainty over the data

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Show how uncertainty in the model translates into uncertainty in expected outcomes.
</div>

We now generate predictions over a grid of storage times and display credible bands.

In [ ]:
cat("generating fitted values...\n")

prediction_grid <- tibble(
  median_rbc_age = seq(min(blood$median_rbc_age), max(blood$median_rbc_age), length.out = 80)
) |>
  mutate(
    median_rbc_age_c = median_rbc_age - mean(blood$median_rbc_age)
  )

fitted_draws <- crossing(
  prediction_grid,
  draw = 1:n_draws
) |>
  mutate(
    .value = beta_draws$b_Intercept[draw] +
      beta_draws$b_median_rbc_age_c[draw] * median_rbc_age_c
  )

fitted_summary <- fitted_draws |>
  group_by(median_rbc_age) |>
  median_qi(.value, .width = c(0.66, 0.95))

cat("fitted values ready.\n")

In [ ]:
ggplot() +
  geom_point(
    data = blood,
    aes(x = median_rbc_age, y = log_preop_psa),
    color = "grey45",
    alpha = 0.65,
    size = 2
  ) +
  geom_ribbon(
    data = fitted_summary |> filter(.width == 0.95),
    aes(x = median_rbc_age, ymin = .lower, ymax = .upper),
    fill = viridis(1, option = "B"),
    alpha = 0.18
  ) +
  geom_ribbon(
    data = fitted_summary |> filter(.width == 0.66),
    aes(x = median_rbc_age, ymin = .lower, ymax = .upper),
    fill = viridis(1, option = "B"),
    alpha = 0.35
  ) +
  geom_line(
    data = fitted_summary |> distinct(median_rbc_age, .value),
    aes(x = median_rbc_age, y = .value),
    color = viridis(1, option = "D"),
    linewidth = 1.1
  ) +
  labs(
    title = "Expected log pre-operative PSA across median RBC age",
    subtitle = "Dark band = 66% credible interval, light band = 95% credible interval",
    x = "Median red blood cell age",
    y = "Expected log(preop_psa)"
  )

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    This figure is often the turning point for learners.
    <br><br>
    Instead of a single fitted line pretending to be certain, we see a <b>range of plausible regression lines</b> condensed into interval bands.
</div>

## 10. Posterior predictive checking

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Check whether data simulated from the fitted model resemble the observed data.
    <br><br>
    A model can produce neat coefficients and still fit poorly. Posterior predictive checks help us inspect that risk.
</div>

In [ ]:
cat("running predictive checks...\n")

set.seed(2026)

draw_ids <- sample(seq_len(n_draws), size = 100)

yrep <- map_dfr(draw_ids, function(i) {
  mu <- beta_draws$b_Intercept[i] +
    beta_draws$b_median_rbc_age_c[i] * blood$median_rbc_age_c

  tibble(
    draw = i,
    yrep = rnorm(nrow(blood), mean = mu, sd = sigma_hat)
  )
})

cat("predictive checks ready.\n")

In [ ]:
ggplot() +
  geom_density(
    data = tibble(y = blood$log_preop_psa),
    aes(x = y),
    color = "black",
    linewidth = 1.2
  ) +
  geom_density(
    data = yrep,
    aes(x = yrep, group = draw),
    color = viridis(1, option = "C"),
    alpha = 0.12
  ) +
  labs(
    title = "Posterior predictive-style density check",
    subtitle = "Black = observed data, coloured lines = replicated datasets",
    x = "log(preop_psa)",
    y = "Density"
  )

In [ ]:
pred_mean_summary <- yrep |>
  group_by(draw) |>
  summarise(rep_mean = mean(yrep), .groups = "drop")

ggplot(pred_mean_summary, aes(x = rep_mean)) +
  geom_histogram(
    bins = 20,
    fill = viridis(1, option = "D"),
    color = "white"
  ) +
  geom_vline(
    xintercept = mean(blood$log_preop_psa),
    color = "firebrick",
    linetype = "dashed",
    linewidth = 1
  ) +
  labs(
    title = "Do replicated datasets reproduce the average outcome?",
    subtitle = "Dashed line = observed sample mean",
    x = "Replicated mean of log(preop_psa)",
    y = "Count"
  )

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Debugging tip:</b> If a Bayesian model seems slow in a classroom setting, reduce <code>n_iter</code> slightly or simplify the formula before changing more advanced Stan controls. A smaller trustworthy model is better than a complicated fragile one.
</div>

## 11. A decision-friendly summary table

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Communicate the most important findings in a polished, non-technical format.
</div>

In [ ]:
cat("building results table...\n")

results_table <- tibble(
  Finding = c(
    "Posterior mean slope",
    "95% credible interval for slope",
    "Probability slope is positive"
  ),
  Value = c(
    round(mean(slope_draws$b_median_rbc_age_c), 4),
    paste0(
      round(quantile(slope_draws$b_median_rbc_age_c, 0.025), 4),
      " to ",
      round(quantile(slope_draws$b_median_rbc_age_c, 0.975), 4)
    ),
    scales::percent(prob_positive, accuracy = 0.1)
  ),
  Interpretation = c(
    "Estimated change in expected log(PSA) for each additional unit of median RBC age",
    "Range of plausible slope values under this model",
    "How strongly the posterior supports a positive association"
  )
)

cat("Bayesian findings at a glance\n")
cat("An uncertainty-aware summary for communication and reporting\n\n")

results_table |>
  rename(`Plain-language meaning` = Interpretation)

cat("\nresults table ready.\n")

## 12. Plain-language interpretation

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Practise turning statistical output into careful scientific language.
</div>

A responsible interpretation of this analysis might look like this:

- The Bayesian regression suggests that haemolysis tends to change with storage time.
- The posterior distribution gives a range of plausible slope values rather than a single all-or-nothing verdict.
- If the posterior probability that the slope is positive is very high, then the model places strong support on a positive association.
- The credible interval shows the uncertainty around the estimated size of that association.
- The posterior predictive checks help us judge whether the model generates data that look reasonably like what we observed.

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    This is one of the biggest strengths of Bayesian analysis in data science teaching:
    <br><br>
    It helps students say <b>how sure</b>, <b>how large</b>, and <b>how variable</b> an effect appears to be.
</div>

## 13. Assumptions, limitations, and responsible use

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Recognise what the model can and cannot tell us.
</div>

This notebook is analytically useful, but it is not magic. Important cautions include:

1. <b>Observational interpretation:</b>  
   This analysis describes association, not guaranteed causation.

2. <b>Model form:</b>  
   We used a simple Gaussian linear regression. If the relationship is strongly nonlinear or the residual structure is unusual, a different model may fit better.

3. <b>Prior assumptions:</b>  
   Bayesian results depend partly on priors, especially in smaller datasets. Here we used weakly informative priors to keep the example stable and teachable.

4. <b>Scope of the data:</b>  
   Our conclusions are limited to the variables included and the observed range of storage times.

5. <b>Prediction vs explanation:</b>  
   A model can explain an average trend reasonably well without being perfect for individual-level prediction.

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> Try one of these safe extensions:
    <ul>
        <li>change <code>ci_width</code> from <code>0.95</code> to <code>0.89</code> and re-run the summary section</li>
        <li>add a quadratic term such as <code>I(rbc_age_c^2)</code> and compare the fitted trend visually</li>
        <li>use <code>conditional_effects(bayes_mod)</code> to inspect the model’s expected pattern through another plotting style</li>
    </ul>
</div>

## 14. Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    You have completed a Bayesian-first workflow in R.
    <br><br>
    You have:
    <ul>
        <li>loaded and prepared a real dataset</li>
        <li>explored a scientifically meaningful relationship</li>
        <li>fitted an interpretable Bayesian model with <code>brms</code></li>
        <li>summarised the posterior distribution</li>
        <li>visualised uncertainty using interval thinking</li>
        <li>checked model behaviour with posterior predictive diagnostics</li>
        <li>communicated findings in polished tables and plain language</li>
    </ul>
    Most importantly, you have seen how Bayesian analysis helps us move beyond binary thinking and toward transparent reasoning under uncertainty.
</div>

## 15. Next steps

If you want to build on this notebook, strong next moves include:

- adding more predictors to create a multiple regression model
- comparing priors and seeing how they influence the posterior
- exploring nonlinear effects
- moving toward multilevel Bayesian models when the teaching context is ready

This exemplar is designed to be a beginning, not an endpoint.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)